# A learning rate for every arm, chosen on the dev split

Report 08 compared five models on SIB-200 and picked their learning rates two different ways:
mmBERT and XLM-R were swept and their best cell quoted, while the from-scratch model and both
untrained controls were run at a single default. That tilted comparisons in **opposite**
directions — against us on ours-vs-mmBERT, in XLM-R's favour on XLM-R-vs-its-own-control.

**Most of that asymmetry has since been closed, and not by this notebook.** Jeffrey's weekend
sweeps took the from-scratch arm to five rates and mmBERT to seven. The cell below recomputes
what is actually on disk instead of restating it here — a pasted table is a cache with no
invalidation, and this notebook's version of that table went stale within a day of being written.

What remains is the half that was always the bigger problem, and it is untouched:

- **Every cell on disk was scored on the same 204 test items it was selected on.**
  `exp_budget_matched_baselines.ipynb` flagged exactly this in its own head-to-head cell —
  *"SIB-200 ships a 99-item validation split, so select there instead before quoting a single
  number in a writeup"* — and nothing has ever used that split. Selecting and reporting on the
  same items inflates every swept arm by an amount nobody here has measured.
- **Both untrained controls are still at one rate each.** So XLM-R's +0.039 over its own control
  is still an upper bound: sweeping a control can only raise it, never lower it.

So this notebook does what is left. **Every arm gets the same seven learning rates, ranked on the
99 dev items, and only the winner is scored on the 204 test items.**

Needs `ft_api` ≥ (1, 4) for `eval_split`. Dev-scored records are tagged `_onval` and
`ft.results()` excludes them by default — a cell selected on the items it is scored on is not a
reportable number, and mixing the two tables is how one ends up on a poster.

In [1]:
import os, sys
REPO = '/content/WashingtonCsed504'
FORK = 'https://github.com/patlkwok/WashingtonCsed504.git'   # YOUR fork, not upstream
if not os.path.exists(REPO):
    !git clone -q {FORK} {REPO}
sys.path.insert(0, f'{REPO}/src/a2-nlp')
import session; factory = session.start(prepare=False)

python 3.12.13 | NVIDIA A100-SXM4-80GB (85 GB, sm_80) | bf16: True
  packages already present
ready — cwd /content/WashingtonCsed504/src/a2-nlp


In [2]:
import importlib, time
import numpy as np
import ft_api as ft
importlib.reload(ft)

assert ft.API_VERSION >= (1, 4), (
    f'ft_api is {ft.API_VERSION}; this notebook needs (1, 4) for eval_split. '
    'Pull the fork and reload -- %autoreload does not work on Python 3.12.')

GPU = ft.gpu_name()
print('ft_api', ft.API_VERSION, '| GPU', GPU)
if 'A100' not in GPU and 'PRO 6000' not in GPU:
    print('  NOTE: every timing quoted here was measured on an A100 or an RTX PRO 6000. This is '
          'neither,\n  so expect them to differ -- an A100 is already 3-4x slower than the '
          'workstation card on these\n  cells, and a T4 is an order of magnitude slower again. '
          'The cost estimate below re-derives\n  itself from records matching this GPU where any '
          'exist.')

sib = ft.load_sib200('yor_Latn')
print('dev items:', len(sib['validation']['text']), '| test items:', len(sib['test']['text']))

ft_api (1, 4) | GPU NVIDIA A100-SXM4-80GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


sib200/yor_Latn: train 701 validation 99 test 204 | 7 classes (chance 0.143) | 1.1% decomposed -> NFC
dev items: 99 | test items: 204


## The asymmetry, measured rather than asserted

Recomputed from `runs/` so it cannot go stale in this markdown the way a pasted table would.

In [3]:
STEPS = 1056

rows = [r for r in ft.results(task='sib200') if r['steps'] == STEPS]
devrows = [r for r in ft.results(task='sib200', eval_split='validation') if r['steps'] == STEPS]

by_test, by_dev = {}, {}
for r in rows:
    by_test.setdefault(r['model_slug'], []).append(r)
for r in devrows:
    by_dev.setdefault(r['model_slug'], []).append(r)

print(f'{"model":<32}{"test LRs":>9}{"dev LRs":>9}{"best":>8}{"at lr":>9}   per-seed s')
for slug, rs in sorted(by_test.items(), key=lambda kv: -max(r['mean'] for r in kv[1])):
    best = max(rs, key=lambda r: r['mean'])
    print(f'{slug:<32}{len(rs):>9}{len(by_dev.get(slug, [])):>9}{best["mean"]:>8.3f}'
          f'{best["lr"]:>9.0e}   {np.mean([r["seconds_per_seed"] for r in rs]):.0f}')

# The 'best' column is best-on-test -- the selection this notebook exists to replace. It is
# printed to be superseded, not quoted.
print(f'\n{len(rows)} test cells, {len(devrows)} dev cells at {STEPS} steps.')
unswept = sorted(s for s, rs in by_test.items() if len(rs) < 3)
if unswept:
    print(f'still at one or two rates: {unswept}')
print('Any arm with 0 dev cells currently has its learning rate chosen on the same 204 items it '
      'is reported on.')

model                            test LRs  dev LRs    best    at lr   per-seed s
swap62k-yor-69.1M-62.5k-s1              1        0   0.705    2e-05   19
swap62k-yor-69.1M-62.5k-s0              1        0   0.693    2e-05   19
swap62k-yor-69.1M-62.5k-s2              1        0   0.677    2e-05   20
yor-64M-62.5k-s0                        5        0   0.666    3e-05   24
yor-64M-62.5k-s2                        1        0   0.656    2e-05   18
yor-64M-46.9k-s0                        1        0   0.647    2e-05   18
yor-16M-62.5k-s0                        1        0   0.645    2e-05   19
yor-64M-62.5k-s1                        1        0   0.644    2e-05   19
yor-4M-62.5k-s0                         1        0   0.636    2e-05   20
multi-yor                               1        0   0.614    2e-05   18
yor-32M-12k-s0                          1        0   0.609    2e-05   18
yor-16M-35.2k-afriberta-s0              1        0   0.603    2e-05   47
yor-16M-11.7k-s0                        1  

## The checkpoints

Three of the five are free. `random_init` and `random_init_like` build an untrained model from a
config in seconds, so both controls regenerate on any runtime. The two hub baselines download.

**The from-scratch checkpoint does not.** `runs/<tag>/` directories are not tracked in git
(`.gitignore` excludes `runs/*/` — they are hundreds of MB), so `yor_64M_62.5k_s0` exists only on the machine that
trained it. Three ways to get it, in order of cost:

1. **Drive** — Jeffrey uploaded it on 9 August: 125 MB, trimmed to `model.safetensors`, both
   configs, `tokenizer.json`, the committed result record and `SHA256SUMS.txt` (the 405 MB
   optimizer state is only useful for resuming pretraining). Set `DRIVE_CKPT` and mount.
   **Much the best option.** Verify `model.safetensors` against `d69657eb…` before sweeping on it.
2. Ask Jeffrey to run this notebook's sweep on the card that already has the checkpoint.
3. **Re-pretrain it here**, at 64M tokens for 62,500 steps — ~40 min on a Blackwell, so
   **1.5–2 h on an A100**, on top of the ~3 h the sweep itself costs there.

**A copy on the laptop is not a copy on the runtime.** The first branch below checks
`factory.RUNS`, which resolves to the *runtime's* disk. A checkpoint sitting in the local clone's
`runs/` — where it belongs, and where `.gitignore` keeps it out of commits — is invisible here.
It has to travel via Drive.

**Re-pretraining does NOT reproduce `yor_64M_62.5k_s0`, and the tag must not say it does.**
Same corpus, same seed and same steps on a different GPU is a different model — nondeterministic
kernels, and this one is a 33.8M model whose seed spread at this cell is 0.103. Three things go
wrong if it inherits the name:

- `runs/yor_64M_62.5k_s0_result.json` is **committed** (val 2.315, 2,409 s on Jeffrey's card).
  `pretrain` would overwrite a scientific record from PR #32 with a run from other hardware.
- The from-scratch **test** rows already exist, measured on Jeffrey's checkpoint.
  `ft.evaluate(reuse=True)` would hand one straight back while the dev sweep ran on the new
  checkpoint — two different models under one name, inside one comparison.
- Nothing would announce either.

So the retrain path writes `yor_64M_62.5k_s0_local`. Distinct pretraining record, distinct
`model_slug`, nothing reused, nothing overwritten — and the comparison stays internally consistent
because *every* cell in it, dev and test, is then measured on the same checkpoint.

The cell fails loudly rather than skipping the arm. A sweep silently missing the one model the
poster's headline rests on is worse than no sweep.

In [4]:
SCRATCH_TAG   = 'yor_64M_62.5k_s0'        # corpus yor, 64M tokens, 62.5k steps, preset poc, seed 0
DRIVE_CKPT    = '/content/drive/MyDrive/csed504-runs/yor_64M_62.5k_s0'
REPRETRAIN    = False                     # True -> retrain it here as _local (1.5-2 h on an A100)
ALLOW_PARTIAL = False                     # True -> sweep the other four arms without it

# The weights the committed downstream rows were measured on. Verifying costs about a second and
# rules out the one thing a path cannot: the expected filename holding a different model.
CKPT_SHA256 = 'd69657ebea1a5aa91f3ca26ae03449ecefabbb4ef85e4adc05ce000921335150'

MMBERT = 'jhu-clsp/mmBERT-base'
XLMR   = 'FacebookAI/xlm-roberta-base'


def find_ckpt(root):
    """The directory holding config.json -- <root>, or the single subdirectory that has it.

    Uploading to Drive nests a level more often than not: a folder dragged into the web UI
    arrives as <root>/<tag>/config.json rather than <root>/config.json. Descending when exactly
    one candidate exists is safe here because the checksum below catches a wrong model even when
    the path looks right.
    """
    if not root or not os.path.isdir(root):
        return None
    if os.path.exists(os.path.join(root, 'config.json')):
        return root
    hits = [os.path.join(root, d) for d in sorted(os.listdir(root))
            if os.path.isdir(os.path.join(root, d))
            and os.path.exists(os.path.join(root, d, 'config.json'))]
    return hits[0] if len(hits) == 1 else None


def why_not(root):
    """Why a DRIVE_CKPT did not resolve. 'Not set' and 'set but unreadable' are different
    problems with different fixes, and reporting them with one message costs a session."""
    out = [f'  DRIVE_CKPT     : {root!r}',
           f'  drive mounted  : {os.path.ismount("/content/drive")}',
           f'  path exists    : {os.path.exists(root)}',
           f'  is a directory : {os.path.isdir(root)}']
    if os.path.isdir(root):
        out.append(f'  contents       : {sorted(os.listdir(root))[:12]}')
        out.append('  -> no config.json here, and not exactly one subdirectory holding one.\n'
                   '     Point DRIVE_CKPT at the directory that directly contains config.json.')
    else:
        parent = os.path.dirname(root.rstrip('/'))
        if os.path.isdir(parent):
            out.append(f'  {parent} holds: {sorted(os.listdir(parent))[:12]}')
        else:
            out.append(f'  parent {parent} does not exist either -- check the path from the top.')
    return '\n'.join(out)


# session.start() does NOT mount Drive -- only save_results() does, and that is the last cell.
# So a DRIVE_CKPT path is simply a missing directory until this runs, which is exactly what the
# old version of this cell reported as "no DRIVE_CKPT was given".
if DRIVE_CKPT and DRIVE_CKPT.startswith('/content/drive') and not os.path.ismount('/content/drive'):
    print('mounting Drive (DRIVE_CKPT points into it and it is not mounted yet)...')
    session.mount_drive()

local = os.path.join(factory.RUNS, SCRATCH_TAG)
found = find_ckpt(DRIVE_CKPT)

if os.path.exists(os.path.join(local, 'config.json')):
    scratch = local
    print(f'from-scratch checkpoint: {scratch} (already on this runtime)')
elif found:
    scratch = found
    print(f'from-scratch checkpoint: {scratch} (from Drive)')
    if found != DRIVE_CKPT:
        print(f'  NOTE: descended into {os.path.relpath(found, DRIVE_CKPT)}/ -- '
              'the upload nested one level.')
elif REPRETRAIN:
    # NOT 'yor_64M_62.5k_s0'. That record is committed and its downstream rows already exist;
    # see the markdown above for what reusing the name would silently do.
    factory = session.start(corpus='yor')                 # verifies fingerprint 15abd33de5af
    rec = factory.pretrain('yor', tokens=64_000_000, steps=62_500, seed=0, preset='poc',
                           tag='yor_64M_62.5k_s0_local')
    scratch = os.path.join(factory.RUNS, rec['tag'])
    print(f'from-scratch checkpoint: {scratch} (RETRAINED HERE, val {rec["val_loss"]:.3f} '
          f'against 2.315 on the workstation)')
    print('  every from-scratch cell below is measured on THIS checkpoint; the committed 0.632 '
          'row is not\n  comparable with them and is excluded from the table.')
elif ALLOW_PARTIAL:
    scratch = None
    print(f'*** {SCRATCH_TAG} NOT FOUND -- sweeping the other four arms without it. ***\n'
          '    The XLM-R-vs-its-own-control correction is complete without this arm; the\n'
          '    ours-vs-mmBERT comparison is NOT, and the table will say so.')
elif DRIVE_CKPT:
    raise FileNotFoundError(
        'DRIVE_CKPT is set, but no checkpoint was found under it.\n\n' + why_not(DRIVE_CKPT) +
        '\n\nA checkpoint directory holds config.json and model.safetensors side by side.\n'
        'Fix the path, or set ALLOW_PARTIAL = True to sweep the other four arms without it.')
else:
    raise FileNotFoundError(
        f'{SCRATCH_TAG} is not on this runtime and DRIVE_CKPT is empty.\n'
        'It is the model the SIB-200 headline rests on, so this notebook will not quietly '
        'produce a table\nwithout it. Set DRIVE_CKPT, or ALLOW_PARTIAL = True to sweep the other '
        'four arms, or REPRETRAIN = True,\nor run this on the card that already has the '
        'checkpoint.')

# Whether the from-scratch arm is the SAME checkpoint the committed rows measured. If it is not,
# those rows describe a different model and the table below must not difference against them.
SCRATCH_IS_CANONICAL = bool(scratch) and not scratch.endswith('_local')

if scratch and SCRATCH_IS_CANONICAL:
    import hashlib
    weights = os.path.join(scratch, 'model.safetensors')
    h = hashlib.sha256()
    with open(weights, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    if h.hexdigest() == CKPT_SHA256:
        print(f'  weights verified against the canonical checkpoint ({h.hexdigest()[:12]}...)')
    else:
        raise ValueError(
            f'{weights}\nhashes to {h.hexdigest()[:12]}... but the canonical checkpoint is '
            f'{CKPT_SHA256[:12]}...\nThis is a DIFFERENT model under the expected name, so its '
            'scores are not comparable with\nthe committed rows. Re-download it, or rename the '
            'directory to end in _local and re-run.')

# Both controls are cheap to rebuild and are byte-identical to the ones already measured.
factory = session.start(corpus='yor')
rand_ours = factory.random_init('yor')
rand_xlmr = factory.random_init_like(XLMR)

ARMS = [('mmBERT',          MMBERT),
        ('XLM-R',           XLMR),
        ('untrained ours',  rand_ours),
        ('untrained XLM-R', rand_xlmr)]
if scratch:
    ARMS.insert(2, ('from-scratch ours', scratch))

print()
for name, path in ARMS:
    print(f'  {name:<20} {path}')
print(f'\n{len(ARMS)} of 5 arms' + ('' if scratch else '  <-- PARTIAL'))

Mounted at /content/drive
from-scratch checkpoint: /content/drive/MyDrive/csed504-runs/yor_64M_62.5k_s0 (from Drive)
  weights verified against the canonical checkpoint (d69657ebea1a...)
python 3.12.13 | NVIDIA A100-SXM4-80GB (85 GB, sm_80) | bf16: True
  packages already present
yor: preparing yor_Latn
  using the shared tokenizer at 'tokenizers/yor-bpe16k' (not training a new one)


README.md:   0%|          | 0.00/329k [00:00<?, ?B/s]

    [fineweb2] 79,999 docs / 260M chars in 61s                        
    encoded 79,999 docs -> 69,596,452 tokens in 66s                        

  decoded sample: '<s> Ẹ̀gbá\nÀwọn àkóónú\nILẸ̀ Ẹ̀GBÁ[àtúnṣe | edit source]\nÓ se pàtàkì láti mọ díẹ̀ nípa ìtàn ilẹ̀ Ẹ̀gbá àti irú ènìyàn tí ń gbé ìlú Ẹ̀gbá. Ìdí èyí ni pé yóò jẹ́'
  69,096,452 train + 500,000 val tokens, 3.73 chars/token
  vocabulary fingerprint 15abd33de5af -- runs only compare across matching fingerprints

  corpus 'yor' ready, tokenizer 15abd33de5af
ready — cwd /content/WashingtonCsed504/src/a2-nlp


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  mmBERT               jhu-clsp/mmBERT-base
  XLM-R                FacebookAI/xlm-roberta-base
  from-scratch ours    /content/drive/MyDrive/csed504-runs/yor_64M_62.5k_s0
  untrained ours       /content/WashingtonCsed504/src/a2-nlp/runs/yor_random_init
  untrained XLM-R      /content/WashingtonCsed504/src/a2-nlp/runs/xlm-roberta-base_random_init

5 of 5 arms


## The dev sweep

**Seven learning rates**, every arm, scored on the 99 dev items. **Three seeds here, not five** —
this pass only has to *rank* the rates; the reported number comes from the separate five-seed run
on test below. Three seeds is the minimum this project accepts for a fine-tuning cell, with the interval
bootstrapped over test items and pooled across seeds rather than taken from the last one.

**Seven rather than the five report 08 used, because five is not wide enough.** mmBERT's best
cell on test sits at **7e-5**, outside a grid that stops at 5e-5. A winner at the end of the
swept range is not a winner — it is evidence the range was too small, and three of five sweeps in
this project have already landed there. Extended for *every* arm rather than only the one that
hit the edge: extending just the arm that ran out of room is the same selection asymmetry this
notebook exists to remove, one level up.

The cell projects its own cost from `seconds_per_seed` on the existing records before it starts,
rather than quoting a number typed into this markdown. Expect **around three hours on an A100**,
under an hour on a Blackwell. `reuse=True` means an interrupted session resumes rather than
restarting, which matters at that length — Colab will end the session before the sweep does.

In [5]:
LRS       = [5e-6, 1e-5, 2e-5, 3e-5, 5e-5, 7e-5, 1e-4]
DEV_SEEDS = (0, 1, 2)
SEEDS     = (0, 1, 2, 3, 4)          # the test pass below; declared here so the estimate can see it
REUSE     = True
QUICK     = False        # True -> 2 LRs, to check the plumbing before committing the full run

lrs = LRS[1:3] if QUICK else LRS

# Project the cost from what this GPU (or the nearest recorded one) actually took, rather than
# from a number typed into the markdown. Same rule as everywhere else here: generate it.
prior = [r for r in ft.results(task='sib200', eval_split=None) if r.get('seconds_per_seed')]
same_gpu = [r for r in prior if r.get('gpu') == GPU]
pool = same_gpu or prior
default_s = float(np.median([r['seconds_per_seed'] for r in pool])) if pool else 60.0
total_s = 0.0
print(f'{"arm":<20}{"s/seed":>8}{"dev":>7}{"test":>7}{"min":>7}   (from {len(pool)} prior cells'
      f'{"" if same_gpu else " -- NONE on this GPU, so this is a guess"})')
for name, path in ARMS:
    cells = [r for r in pool if r['model'] == path]
    s = float(np.median([r['seconds_per_seed'] for r in cells])) if cells else default_s
    arm_s = s * (len(lrs) * len(DEV_SEEDS) + len(SEEDS))
    total_s += arm_s
    print(f'{name:<20}{s:>8.0f}{len(lrs) * len(DEV_SEEDS):>7}{len(SEEDS):>7}{arm_s / 60:>7.0f}'
          f'{"" if cells else "   <- no prior cell, assumed"}')
print(f'\nprojected total: {total_s / 60:.0f} min ({total_s / 3600:.1f} h) for '
      f'{len(ARMS) * (len(lrs) * len(DEV_SEEDS) + len(SEEDS))} fine-tuning runs.')
print('reuse=True: anything already on disk is skipped, so a resumed session costs only what is '
      'left.')

t0 = time.time()
for name, path in ARMS:
    print(f'\n{name}')
    for lr in lrs:
        ft.evaluate(path, task='sib200', lr=lr, steps=STEPS, seeds=DEV_SEEDS,
                    data=sib, reuse=REUSE, label=f'{name} lr{lr:g}',
                    eval_split='validation')

print(f'\ndev sweep: {(time.time() - t0) / 60:.1f} min')

arm                   s/seed    dev   test    min   (from 18 prior cells)
mmBERT                   122     21      5     53
XLM-R                     64     21      5     28
from-scratch ours         95     21      5     41   <- no prior cell, assumed
untrained ours            20     21      5      9
untrained XLM-R           95     21      5     41   <- no prior cell, assumed

projected total: 172 min (2.9 h) for 130 fine-tuning runs.
reuse=True: anything already on disk is skipped, so a resumed session costs only what is left.

mmBERT


config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: /content/WashingtonCsed504/src/a2-nlp/runs/xlm-roberta-base_random_init
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: /content/WashingtonCsed504/src/a2-nlp/runs/xlm-roberta-base_random_init
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  untrained XLM-R lr0.0001   macro_f1 0.091 +/-0.047 (seed sd) | 95% CI [0.069, 0.108] | n_train 701 | 95s/seed
    AT OR BELOW CHANCE (0.143) — this cell did not learn the task. Per-seed [0.058, 0.058, 0.157].
    Do not read it as a coverage result, and do not compare it with anything: a collapsed run
    moves further between reruns than any effect being tested.

dev sweep: 135.5 min


## Select on dev, report on test

The winner of each arm — chosen on the 99 dev items it will *not* be reported on — is then run at
five seeds on test. Where that cell already exists from earlier work it is reused, so this pass is
short.

In [6]:
dev = ft.results(task='sib200', eval_split='validation')
picked, at_edge = {}, []
for name, path in ARMS:
    rs = [r for r in dev if r['model'] == path and r['steps'] == STEPS]
    if not rs:
        print(f'{name}: no dev cells -- run the sweep above'); continue
    win = max(rs, key=lambda r: r['mean'])
    picked[name] = (path, win['lr'], win['mean'])
    ranked = ', '.join(f'{r["lr"]:g}:{r["mean"]:.3f}' for r in sorted(rs, key=lambda r: -r['mean']))
    edge = win['lr'] in (min(lrs), max(lrs))
    if edge:
        at_edge.append((name, win['lr']))
    print(f'{name:<20} dev pick lr {win["lr"]:g}  ({ranked})'
          f'{"   <-- AT THE GRID EDGE" if edge else ""}')

# A winner at either end of the swept range means the optimum is probably outside it, so the
# cell reports a boundary rather than a best. This is what made three earlier sweeps in this
# project unquotable, and it is the reason the grid was widened to seven rates in the first place.
if at_edge:
    print('\n*** ' + '; '.join(f'{n} picked {lr:g}' for n, lr in at_edge) + ' -- at the grid edge.')
    print('    Extend LRS past it and re-run FOR EVERY ARM, not only the ones that hit it.')
    print('    Until then those rows state where the grid stopped, not where the optimum is.')

print()
final = {}
for name, (path, lr, _) in picked.items():
    final[name] = ft.evaluate(path, task='sib200', lr=lr, steps=STEPS, seeds=SEEDS,
                              data=sib, reuse=REUSE, label=name)

mmBERT               dev pick lr 7e-05  (7e-05:0.586, 5e-05:0.552, 0.0001:0.537, 3e-05:0.514, 2e-05:0.504, 1e-05:0.469, 5e-06:0.429)
XLM-R                dev pick lr 3e-05  (3e-05:0.453, 1e-05:0.403, 2e-05:0.362, 5e-06:0.303, 0.0001:0.058, 5e-05:0.058, 7e-05:0.058)
from-scratch ours    dev pick lr 3e-05  (3e-05:0.635, 0.0001:0.628, 5e-05:0.622, 2e-05:0.618, 7e-05:0.600, 1e-05:0.458, 5e-06:0.282)
untrained ours       dev pick lr 0.0001  (0.0001:0.415, 7e-05:0.405, 2e-05:0.381, 3e-05:0.378, 5e-05:0.377, 1e-05:0.121, 5e-06:0.058)   <-- AT THE GRID EDGE
untrained XLM-R      dev pick lr 3e-05  (3e-05:0.386, 2e-05:0.366, 1e-05:0.319, 5e-05:0.235, 5e-06:0.163, 7e-05:0.094, 0.0001:0.091)

*** untrained ours picked 0.0001 -- at the grid edge.
    Extend LRS past it and re-run FOR EVERY ARM, not only the ones that hit it.
    Until then those rows state where the grid stopped, not where the optimum is.



Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/mmBERT-base
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/mmBERT-base
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: /content/WashingtonCsed504/src/a2-nlp/runs/xlm-roberta-base_random_init
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: /content/WashingtonCsed504/src/a2-nlp/runs/xlm-roberta-base_random_init
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  untrained XLM-R            macro_f1 0.382 +/-0.017 (seed sd) | 95% CI [0.330, 0.425] | n_train 701 | 95s/seed


## The symmetric table

Every row now chosen the same way, on data it is not scored on. The last column is what changed
against report 08, whose rows were selected two different ways.

Then a second, smaller table: **dev-selected against best-on-test, per arm.** That difference is
the optimism in the old procedure — what picking the maximum over a grid and reporting it on the
same 204 items was worth. Report 08 and the poster both currently quote best-on-test numbers, so
this is the correction factor for every SIB-200 figure in them, measured rather than argued.

In [7]:
FLOOR = 0.06     # smallest macro-F1 difference 204 test items resolve; see reports/06

R08 = {'from-scratch ours': 0.6324, 'mmBERT': 0.5736, 'XLM-R': 0.4077,
       'untrained ours': 0.4034, 'untrained XLM-R': 0.3692}
if not SCRATCH_IS_CANONICAL:
    # A different checkpoint. Differencing against 0.632 would report a hardware difference and
    # a selection change as if they were one number.
    del R08['from-scratch ours']

print(f'{"model":<20}{"lr":>8}{"macro-F1":>10}{"sd":>7}   {"95% CI":<18}{"report 08":>10}{"move":>8}')
order = sorted(final, key=lambda k: -final[k]['mean'])
for name in order:
    r = final[name]
    ci = f'[{r["ci"][0]:.3f}, {r["ci"][1]:.3f}]'
    old = R08.get(name)
    cmp = f'{old:>10.3f}{r["mean"] - old:>+8.3f}' if old else f'{"n/a":>10}{"--":>8}'
    print(f'{name:<20}{r["lr"]:>8.0e}{r["mean"]:>10.3f}{r["sd"]:>7.3f}   {ci:<18}{cmp}')
if scratch and not SCRATCH_IS_CANONICAL:
    print('\nfrom-scratch was retrained on this runtime, so it has no report-08 comparison: '
          'that row\nmeasured a different checkpoint. The within-table gaps below are still valid.')
if not scratch:
    print('\n*** PARTIAL: the from-scratch arm is missing (no checkpoint on this runtime). ***\n'
          '    Complete here: XLM-R against its own untrained control, symmetric and dev-selected.\n'
          '    Still open:    ours against mmBERT. Do not quote a headline from this table.')
if at_edge:
    print('\nSelected at the edge of the swept range, so these are boundaries rather than optima: '
          + ', '.join(f'{n} ({lr:g})' for n, lr in at_edge)
          + '.\nAn optimum outside the grid can only RAISE that arm, so any gap measured over it '
            'is an upper\nbound. Comparisons not involving these arms are unaffected.')

# What the SELECTION RULE alone was worth. Best-on-test is how every row on disk was chosen
# before today; dev-selected is how they are chosen now.
#
# Match on the SLUG, not on r['model']. The model field is a filesystem path and it differs by
# machine for the same weights -- the committed from-scratch rows say 'runs/yor_64M_62.5k_s0'
# while this session's say '/content/drive/.../yor_64M_62.5k_s0', and yor_random_init is already
# on disk under two different paths from two machines. Matching on the path found nothing for
# three of the five arms and dropped them silently, which is this notebook's own subject matter:
# a comparison that quietly compares fewer things than it claims to.
# The slug is also narrower than a substring match -- it separates yor-64M-62.5k-s0 from s1, s2
# and 46.9k, so 'best on test' cannot wander onto a different checkpoint.
arm_slug = {name: ft._slug(path) for name, path in ARMS}
test_rows = [r for r in ft.results(task='sib200') if r['steps'] == STEPS]

print(f'\n{"model":<20}{"dev-selected":>14}{"best-on-test":>14}{"optimism":>10}   at lr')
missing = []
for name in order:
    cand = [r for r in test_rows if r['model_slug'] == arm_slug[name]]
    if not cand:
        missing.append(name)
        continue
    bt = max(cand, key=lambda r: r['mean'])
    print(f'{name:<20}{final[name]["mean"]:>14.3f}{bt["mean"]:>14.3f}'
          f'{bt["mean"] - final[name]["mean"]:>+10.3f}   {bt["lr"]:.0e}')
if missing:
    print(f'\n*** no test records found for: {missing} -- this table is INCOMPLETE. ***')
print('\noptimism = what best-on-test claims over the cell chosen on data it is not scored on.\n'
      'It is a property of the selection rule, not of the model. Non-negative by construction,\n'
      'since the dev-selected cell is itself one of the candidates; what matters is its size\n'
      'against the 0.06 floor.')


def gap(a, b):
    """Difference, with both gates the project requires: the floor and CI overlap."""
    ra, rb = final[a], final[b]
    d = ra['mean'] - rb['mean']
    overlap = ra['ci'][0] <= rb['ci'][1] and rb['ci'][0] <= ra['ci'][1]
    verdict = ('below the 0.06 floor' if abs(d) < FLOOR else
               'CIs overlap' if overlap else 'clears both gates')
    print(f'{a} - {b}: {d:+.3f}  ({verdict})')


print()
for a, b in [('from-scratch ours', 'mmBERT'),
             ('XLM-R', 'untrained XLM-R'),
             ('from-scratch ours', 'untrained ours')]:
    if a in final and b in final:
        gap(a, b)

model                     lr  macro-F1     sd   95% CI             report 08    move
from-scratch ours      3e-05     0.688  0.024   [0.631, 0.734]         0.632  +0.056
mmBERT                 7e-05     0.582  0.023   [0.518, 0.635]         0.574  +0.009
untrained ours         1e-04     0.429  0.036   [0.379, 0.468]         0.403  +0.025
untrained XLM-R        3e-05     0.382  0.017   [0.330, 0.425]         0.369  +0.013
XLM-R                  3e-05     0.358  0.161   [0.311, 0.398]         0.408  -0.049

Selected at the edge of the swept range, so these are boundaries rather than optima: untrained ours (0.0001).
An optimum outside the grid can only RAISE that arm, so any gap measured over it is an upper
bound. Comparisons not involving these arms are unaffected.

model                 dev-selected  best-on-test  optimism   at lr
from-scratch ours            0.688         0.688    +0.000   3e-05
mmBERT                       0.582         0.582    +0.000   7e-05
untrained ours          

## Per-seed scores, before anything above is quoted

XLM-R's cell has a seed sd of **0.161**; every other arm is between 0.017 and 0.036. In this
project that gap has always meant one thing — a cell where some seeds trained and some did not,
whose mean describes no run that actually happened.

The dev sweep says where it comes from: XLM-R scored **0.058 at 5e-5, 7e-5 and 1e-4**, below the
0.143 chance line, so the selected 3e-5 sits directly against a cliff.

Note what the interval does here. XLM-R's 95% CI is *narrower* than its seed spread implies,
because the bootstrap resamples test items with predictions pooled across seeds — it cannot see
variation between seeds at all. Neither the CI nor a glance at the mean catches this; only the
per-seed scores and the chance line do.

In [9]:
CHANCE = 1 / len(sib['labels'])

print(f'{"arm":<20}{"mean":>7}{"sd":>7}   per-seed')
for name in order:
    r = final[name]
    seeds = ', '.join(f'{s:.3f}' for s in r.get('scores', []))
    print(f'{name:<20}{r["mean"]:>7.3f}{r["sd"]:>7.3f}   {seeds}')

# A cell whose seeds straddle chance is a mixture, not a measurement. Report it as a fraction
# that trained -- the same rule the 98M pretraining runs needed, where some seeds left the
# unigram plateau and some never did, and the average described neither population.
print()
for name in order:
    s = final[name].get('scores', [])
    below = [x for x in s if x < CHANCE]
    if below and len(below) < len(s):
        print(f'*** {name}: {len(below)} of {len(s)} seeds below chance ({CHANCE:.3f}), '
              f'the rest above.')
        print(f'    Quote it as "{len(s) - len(below)} of {len(s)} seeds trained", not as '
              f'{final[name]["mean"]:.3f}.')
    elif s and max(s) - min(s) > 0.2:
        print(f'*** {name}: seeds span {max(s) - min(s):.3f} without crossing chance -- unstable '
              f'but not degenerate.\n    Say so next to the mean.')

arm                    mean     sd   per-seed
from-scratch ours     0.688  0.024   0.667, 0.695, 0.655, 0.712, 0.712
mmBERT                0.582  0.023   0.590, 0.586, 0.538, 0.594, 0.604
untrained ours        0.429  0.036   0.408, 0.372, 0.468, 0.430, 0.466
untrained XLM-R       0.382  0.017   0.396, 0.377, 0.394, 0.395, 0.350
XLM-R                 0.358  0.161   0.543, 0.386, 0.409, 0.396, 0.057

*** XLM-R: 1 of 5 seeds below chance (0.143), the rest above.
    Quote it as "4 of 5 seeds trained", not as 0.358.


## Extending the grid past the edge

`untrained ours` selected **1e-4**, the top of the seven-rate grid, and was still climbing
(1e-4: 0.415, 7e-5: 0.405). Its optimum is therefore outside the range and its score is a lower
bound, which makes every gap measured over it an upper bound.

It changes no comparison in the table above — the headline is ours (3e-5) against mmBERT (7e-5),
both interior, and the XLM-R contrast is 3e-5 against 3e-5, also interior. But that arm is the
SIB-200 **floor**, the floor is poster material, and an unswept boundary is exactly the defect
we flagged in the NER floor this week. Cheaper to fix it than to caveat it.

**Extended for all five arms, not only the one that ran out of room.** Every other arm has a
demonstrated interior peak with a declining tail, so widening cannot help them and the extension
hands nobody extra chances to win. Doing it symmetrically anyway costs about forty minutes and
removes the argument entirely, which is the position we have twice asked others on this project
to take.

`reuse=True` skips the cells already on disk, so only the 30 new ones cost anything.

In [10]:
LRS_EXT  = [2e-4, 3e-4]
lrs_full = lrs + LRS_EXT

# Project from what this session actually measured, not from the markdown above.
prior = [r for r in ft.results(task='sib200', eval_split=None) if r.get('seconds_per_seed')]
est = 0.0
for name, path in ARMS:
    cells = [r for r in prior if r['model_slug'] == ft._slug(path)]
    if cells:
        est += float(np.median([r['seconds_per_seed'] for r in cells])) \
               * len(LRS_EXT) * len(DEV_SEEDS)
print(f'{len(ARMS) * len(LRS_EXT) * len(DEV_SEEDS)} new dev runs, projected {est / 60:.0f} min '
      f'(plus a 5-seed test cell for any arm whose pick moves)\n')

t0 = time.time()
for name, path in ARMS:
    print(f'\n{name}')
    for lr in LRS_EXT:
        ft.evaluate(path, task='sib200', lr=lr, steps=STEPS, seeds=DEV_SEEDS,
                    data=sib, reuse=REUSE, label=f'{name} lr{lr:g}',
                    eval_split='validation')

print(f'\nextension: {(time.time() - t0) / 60:.1f} min')

30 new dev runs, projected 37 min (plus a 5-seed test cell for any arm whose pick moves)


mmBERT


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/mmBERT-base
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/mmBERT-base
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: /content/WashingtonCsed504/src/a2-nlp/runs/xlm-roberta-base_random_init
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: /content/WashingtonCsed504/src/a2-nlp/runs/xlm-roberta-base_random_init
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  untrained XLM-R lr0.0003   macro_f1 0.058 +/-0.000 (seed sd) | 95% CI [0.042, 0.072] | n_train 701 | 95s/seed
    AT OR BELOW CHANCE (0.143) — this cell did not learn the task. Per-seed [0.058, 0.058, 0.058].
    Do not read it as a coverage result, and do not compare it with anything: a collapsed run
    moves further between reruns than any effect being tested.

extension: 38.6 min


## The final table — nine rates, every arm selected the same way

**This supersedes the table above.** That one was computed on the seven-rate grid, where one arm
selected at its boundary; this one re-selects across all nine rates. Both are kept deliberately,
because the sequence is the point: the guard fired, the grid was widened, and the answer either
moved or it did not.

Only arms whose dev pick actually changes are re-scored on test — `reuse=True` returns the
existing five-seed record for the rest, so nothing else is recomputed and nothing is overwritten.

In [11]:
arm_slug = {name: ft._slug(path) for name, path in ARMS}
dev2 = ft.results(task='sib200', eval_split='validation')

picked2, at_edge2 = {}, []
for name, path in ARMS:
    rs = [r for r in dev2 if r['model_slug'] == arm_slug[name] and r['steps'] == STEPS]
    if not rs:
        print(f'{name}: no dev cells'); continue
    win = max(rs, key=lambda r: r['mean'])
    picked2[name] = (path, win['lr'], win['mean'])
    was = picked.get(name, (None, None, None))[1]
    edge = win['lr'] in (min(lrs_full), max(lrs_full))
    if edge:
        at_edge2.append((name, win['lr']))
    ranked = ', '.join(f'{r["lr"]:g}:{r["mean"]:.3f}'
                       for r in sorted(rs, key=lambda r: -r['mean']))
    tail = ('   <-- STILL AT THE EDGE' if edge else
            '' if was == win['lr'] else
            f'   <-- moved from {was:g}' if was is not None else '   <-- new arm')
    print(f'{name:<20} dev pick lr {win["lr"]:g}  ({ranked}){tail}')

print()
final2 = {}
for name, (path, lr, _) in picked2.items():
    final2[name] = ft.evaluate(path, task='sib200', lr=lr, steps=STEPS, seeds=SEEDS,
                               data=sib, reuse=REUSE, label=name)

print(f'\n{"model":<20}{"lr":>8}{"macro-F1":>10}{"sd":>7}   {"95% CI":<18}'
      f'{"report 08":>10}{"move":>8}')
order2 = sorted(final2, key=lambda k: -final2[k]['mean'])
for name in order2:
    r = final2[name]
    ci = f'[{r["ci"][0]:.3f}, {r["ci"][1]:.3f}]'
    old = R08.get(name)
    cmp = f'{old:>10.3f}{r["mean"] - old:>+8.3f}' if old else f'{"n/a":>10}{"--":>8}'
    print(f'{name:<20}{r["lr"]:>8.0e}{r["mean"]:>10.3f}{r["sd"]:>7.3f}   {ci:<18}{cmp}')
print('\n"move" is against report 08, whose rows differ from these in BOTH learning rate and '
      'seed count\n(3 there, 5 here). It is a change of procedure, not a change in any model.')

if at_edge2:
    print('\nStill at the edge after extending: '
          + ', '.join(f'{n} ({lr:g})' for n, lr in at_edge2)
          + '.\nReport those as lower bounds. Do not widen a third time without a reason beyond '
            'the boundary itself.')
else:
    print('\nEvery arm now selects strictly inside the grid -- no boundary caveat on any row.')

test_rows = [r for r in ft.results(task='sib200') if r['steps'] == STEPS]
print(f'\n{"model":<20}{"dev-selected":>14}{"best-on-test":>14}{"delta":>8}   at lr')
for name in order2:
    cand = [r for r in test_rows if r['model_slug'] == arm_slug[name]]
    if not cand:
        print(f'{name:<20}     -- no test records found --'); continue
    bt = max(cand, key=lambda r: r['mean'])
    print(f'{name:<20}{final2[name]["mean"]:>14.3f}{bt["mean"]:>14.3f}'
          f'{bt["mean"] - final2[name]["mean"]:>+8.3f}   {bt["lr"]:.0e}')
print('delta is what best-on-test claims over the dev-selected cell. It is bounded below by zero\n'
      'only because the dev-selected cell is itself a candidate, so read it as "did dev and test\n'
      'agree on the argmax", not as a clean estimate of selection inflation.')

print()
for a, b in [('from-scratch ours', 'mmBERT'),
             ('XLM-R', 'untrained XLM-R'),
             ('from-scratch ours', 'untrained ours')]:
    if a in final2 and b in final2:
        ra, rb = final2[a], final2[b]
        d = ra['mean'] - rb['mean']
        lo, hi = max(ra['ci'][0], rb['ci'][0]), min(ra['ci'][1], rb['ci'][1])
        verdict = ('below the 0.06 floor' if abs(d) < FLOOR else
                   f'CIs overlap by {hi - lo:.3f}' if lo <= hi else 'clears both gates')
        print(f'{a} - {b}: {d:+.3f}  ({verdict})')

mmBERT               dev pick lr 7e-05  (7e-05:0.586, 5e-05:0.552, 0.0002:0.541, 0.0001:0.537, 3e-05:0.514, 2e-05:0.504, 1e-05:0.469, 0.0003:0.434, 5e-06:0.429)
XLM-R                dev pick lr 3e-05  (3e-05:0.453, 1e-05:0.403, 2e-05:0.362, 5e-06:0.303, 0.0001:0.058, 0.0002:0.058, 0.0003:0.058, 5e-05:0.058, 7e-05:0.058)
from-scratch ours    dev pick lr 3e-05  (3e-05:0.635, 0.0001:0.628, 5e-05:0.622, 2e-05:0.618, 0.0002:0.602, 7e-05:0.600, 1e-05:0.458, 0.0003:0.425, 5e-06:0.282)
untrained ours       dev pick lr 0.0001  (0.0001:0.415, 7e-05:0.405, 2e-05:0.381, 3e-05:0.378, 5e-05:0.377, 0.0002:0.361, 0.0003:0.154, 1e-05:0.121, 5e-06:0.058)
untrained XLM-R      dev pick lr 3e-05  (3e-05:0.386, 2e-05:0.366, 1e-05:0.319, 5e-05:0.235, 5e-06:0.163, 7e-05:0.094, 0.0001:0.091, 0.0002:0.058, 0.0003:0.058)

  mmBERT                     0.582 (reusing record; reuse=False to rerun)
  XLM-R                      0.358 (reusing record; reuse=False to rerun)
  from-scratch ours          0.688 (reusing r

## Reading it

Three things this can show, and they are not equally good for the study.

**If our model stays ahead of mmBERT.** The headline survives a symmetric comparison, and it is
no longer open to "you swept theirs and not yours". Note the margin was 0.059 in report 08 —
already *under* the 0.06 floor, so the honest wording is "ahead, not distinguishable", whatever
the sweep does to it.

**If XLM-R's +0.039 over its own control shrinks.** Expected, since only XLM-R was swept before.
It sharpens report 08's claim rather than damaging it: whatever XLM-R learned from 100 languages
does not reach Yoruba. If it shrinks to nothing, say that.

**If our model's win depends on the learning rate it was given.** The uncomfortable outcome, and
the reason to run this rather than assume. A default inherited from `FT_LR_SCRATCH` deciding the
study's headline would be the same failure this project has now caught ten times — a constant
chosen for one context deciding a result in another. Better found here than on the poster.

Whatever comes out, the dev-selected numbers are the ones to quote, and report 08's table needs
the learning rate and the selection rule stated next to every row.

In [12]:
session.save_results()   # asserts Drive is really mounted before copying

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
copied 613 result files -> /content/drive/MyDrive/csed504-runs


613